# New Work-In-Progress

An examination of the likelihood of an AI-caused human extinction.

## Me

Extreme value statistics are used to predict return times for rare events based on events that have happened in the past, but is there a type of statistical analysis, perhaps Bayesian in nature, for unlikely events that have never occurred?

## Claude

Good question — extreme value theory (EVT) already stretches into "not yet observed" territory (that's the whole point of fitting a GEV or GPD and extrapolating past the largest observed value), but you're right that it still assumes the event is drawn from the *same generating process* as your data, just further out on the tail. For something that's never happened at all — a genuinely novel category of event — you need different tools. A few families:

**1. Bayesian extensions of EVT itself.** Instead of point-estimating the GEV/GPD parameters from historical maxima, you put priors on them (often informed by physical theory or analogous systems) and get a posterior predictive distribution for events beyond anything observed. This lets you extrapolate with honest uncertainty rather than a single fitted curve, but it's still an extrapolation of an existing process — not a new category.

**2. Structural / mechanistic risk models**, used heavily in nuclear and aerospace safety: probabilistic risk assessment (PRA) with fault trees and event trees. Instead of asking "how often has a core meltdown happened," you decompose the system into components with known (or estimated) failure rates and propagate the probabilities through the causal structure. This is exactly how you assign a nonzero probability to something with zero historical occurrences — Feynman's critique of NASA after Challenger was precisely that engineers' bottom-up failure-mode estimates were being overridden by "well, it's never failed before."

**3. Reference-class / outside-view forecasting.** If the event has never happened *to you* but has happened to relevantly similar systems, you borrow their base rate. This is Kahneman's "outside view," and it's the same logic behind using paleobiological species-duration data to bound how long a species like *Homo sapiens* might last, even though no human extinction event is in the historical record.

**4. Zero-event Bayesian inference.** If you've had n trials/years/flights with zero occurrences, Laplace's rule of succession (or a Jeffreys prior) gives you a principled upper bound on the rate rather than naively concluding probability zero — the "rule of three" (roughly 3/n for a 95% upper bound) is the frequentist shadow of this.

**5. Anthropic / self-sampling arguments**, for genuinely singular events like human extinction: the Doomsday Argument and Gott's delta-t rule reason from your position in a sequence rather than from any observed frequency of the event itself.

**6. Structured expert elicitation** (Cooke's Classical Model is the most rigorous version), which formally combines and calibrates expert judgment when no data exists at all, often folded into a Bayesian hierarchical model alongside whatever weak empirical priors you do have.

**7. Imprecise/robust Bayes**, for cases where you don't trust any single prior enough to commit to it — you carry a *set* of priors forward and report the resulting interval of posteriors, which is standard in deep-uncertainty and catastrophe modeling.

This is actually almost exactly the toolkit you've already assembled in your human-time-remaining project — Gott's argument and the Carter-Leslie Doomsday reasoning for the never-happened-yet extinction event, paleobiological mammal durations as the reference-class/outside-view estimate, and Einmahl et al.'s EVT approach as the tail-extrapolation angle. If you wanted to extend that project further, the PRA/fault-tree angle (decomposing extinction into causal pathways — pandemic, asteroid, ecological collapse — each with its own hazard model) or a formal robust-Bayes treatment across your three reference classes would be the natural next additions.

## Me

If we did extend that project further, using the PRA/fault-tree angle (decomposing extinction into causal pathways — pandemic, asteroid, ecological collapse — each with its own hazard model) and also include a formal robust-Bayes treatment across the three reference classes, could you elaborate on how we might do that?

## Claude

## 1. PRA / fault-tree decomposition

One design decision to flag first: your original scoping explicitly *excluded* "sudden catastrophes (war, abrupt climate collapse)" so that Part A/B/C could isolate a background evolutionary/ecological timescale. Adding pandemic, asteroid, and ecological-collapse pathways deliberately reintroduces exactly that category — which is fine and is a natural next layer, but it means the cleanest structure is **competing risks**: a background hazard (from your reference-class survival curves) running in parallel with a catastrophe hazard (from the fault tree), rather than folding catastrophe risk into the reference-class analysis itself.

**Top event and combination logic.** Define the top event as "extinction by time T," and treat overall survival as the product of independent-pathway survival functions unless you want to model shared root causes:

S(t) = S_background(t) · S_pandemic(t) · S_asteroid(t) · S_ecological(t)

A pure product tree (OR-gate across pathways, each pathway independent) is the reasonable first pass. A more realistic version would add AND/OR structure for shared triggers — e.g. ecological collapse can be *caused by* a pandemic-driven societal breakdown or an asteroid winter — but that's a refinement, not a prerequisite.

**Per-pathway hazard models**, roughly in order of how much real data you have to lean on:

- **Asteroid/comet impact** is the most tractable: NASA/ESA near-Earth-object survey data gives a size-frequency distribution, and crater-count records extend that empirically — this is a genuine EVT-style extrapolation, size-dependent rate combined with a size-dependent "civilization-ending" threshold (Chicxulub-scale vs. Tunguska/Chelyabinsk-scale). Model as a marked Poisson process: rate λ(size) from survey/crater data, lethality probability p(size) as the mark.
- **Pandemic** has historical frequency data (plague, 1918 flu, COVID) for "pandemic events" but zero historical events at extinction-level severity — a good use case for the zero-event Bayesian inference we discussed (Laplace's rule / Jeffreys prior) layered under a hierarchical model with parameters like emergence rate, R0, case-fatality rate, and intervention effectiveness. Toby Ord's *The Precipice* has usable elicited probability estimates if you want an informative prior rather than starting from Jeffreys.
- **Ecological collapse** is the least quantifiable — probably best treated as a time-*varying* hazard tied to an anthropogenic driver trajectory (e.g. a logistic-increasing hazard keyed to warming or biodiversity-loss trend lines) rather than a constant-rate process, since the physical picture is that the risk is rising, not flat. This one likely needs structured expert elicitation for its priors rather than raw frequency data.

Each of these fits naturally as its own PyMC sub-model producing a posterior hazard function, then you combine them via the survival-function product above and propagate uncertainty by sampling from all sub-models jointly rather than plugging in point estimates.

## 2. Robust Bayes across the three reference classes

Right now you likely have (implicitly) three separate posteriors — one per reference class (All Mammalia, Primates, Large-bodied), each further split by parametric form (Kaplan-Meier empirical vs. log-logistic). A formal robust-Bayes treatment means refusing to collapse that into one weighted average, and instead reporting how far your conclusions move as you vary *which* prior/reference-class you trust:

- **Define a prior class Γ** = the set of all (reference-class, parametric-form) posteriors you've already built. For any quantity of interest — say P(T < 1 Myr) — compute the **lower and upper bound** across every element of Γ. That interval, not a point estimate, is your robust answer. This is the classic Berger-style "class of priors" sensitivity analysis, and you already have the ingredients since you've built all three posteriors.
- This is deliberately different from **Bayesian model averaging**, which would assign weights to the three reference classes (e.g. by phylogenetic similarity to humans, or by out-of-sample predictive fit via WAIC/LOO) and blend them into a single posterior. BMA is easier to compute and gives a tidier single distribution, but it hides the structural disagreement between reference classes inside a weighting choice you may not actually be confident in. Robust Bayes keeps that disagreement visible as an interval width.
- A middle path, easy to implement in PyMC: run the hierarchical model with reference-class as a categorical hyperparameter with a weakly-informative hyperprior (soft BMA), *and separately* report the robust min/max interval across the unweighted set — showing both "our best single estimate" and "how much that estimate depends on which reference class you believe."

**Putting both pieces together**, the full pipeline becomes: sample background survival curves under each (reference-class, form) pair → sample the three pathway hazard sub-models → combine into a per-scenario total survival function → report both a central (BMA-weighted) time-remaining posterior and a robust interval spanning the reference-class choices. That gives you three layers of stated uncertainty (statistical, reference-class/structural, and pathway-model) instead of collapsing everything into one number, which fits the spirit of what your v1/v2 notebooks have already been doing with the outside-view/Gott/Carter-Leslie comparison.

Happy to sketch this out as a v3 notebook outline, or start with just one piece (e.g. the asteroid fault-tree sub-model, since it has the most solid data) if you'd rather build incrementally.

## Me

I'd like to start out by having you sketch this out as a v3 notebook outline.

## Claude

The outline follows your v1/v2 conventions (abstract, pros/cons, conclusions, references) and adds Part D (the fault tree), Part E (combining it with the background hazard from Parts A–C), and Part F (the robust-Bayes treatment).

One thing worth deciding before you build it: Part D reintroduces exactly the "sudden catastrophe" category your original scoping excluded, so I structured it as a separate foreground hazard multiplied against the existing background survival curves rather than folding it into the reference-class analysis — that keeps the two kinds of risk from getting conflated, but it's a framing choice you may want to push back on or adjust. The asteroid sub-model is the best-anchored one to build first since it has real survey data behind it; pandemic and ecological-collapse are progressively more dependent on subjective priors, which the outline calls out explicitly rather than hiding in a parametric choice.

## `human_time_remaining_v3.ipynb` — Outline

**Working title:** Catastrophe Pathway Fault Tree + Robust-Bayes Reference-Class Synthesis

This extends v2 (three reference-class outside-view estimates + Gott delta-t + Carter-Leslie)
with (1) an explicit fault tree for catastrophic extinction pathways, and (2) a formal
robust-Bayes treatment across the reference classes instead of picking or softly averaging one.

---

## 0. Abstract
Same convention as v1/v2: 1 paragraph stating what's new in this notebook relative to v2,
and the headline result once you have it (e.g. "adding catastrophe pathways shifts the
combined median from X to Y; robust bounds across reference classes span [lo, hi]").

## 1. Recap of v1/v2 state (short, mostly loaded not re-derived)
- Reload the three reference-class posteriors from v2: All Mammalia (median 5.2 Myr),
  Primates (median 3.5 Myr), Large-bodied ≥10kg (median 3.9 Myr)
- Reload Gott delta-t and Carter-Leslie posteriors (using N_past ≈ 79.4B, 95% CI [70.5B, 89.4B])
- One paragraph noting the framing shift: v1/v2 modeled a *background* hazard (evolutionary/
  ecological turnover, "no privileged position" reasoning); v3 adds a *foreground* hazard
  (acute catastrophe) that was explicitly excluded from that scope. State this framing
  change explicitly in the notebook so the two hazard types aren't implicitly conflated.

## 2. Part D — Catastrophe fault tree

### 2.1 Tree structure
- Top event: "extinction-causing catastrophe by time *t*"
- First pass: OR-gate, pathways treated as independent competing risks
- Note (for a v4, not implemented here): shared-cause refinement — ecological collapse can
  be triggered by pandemic-driven societal collapse or an asteroid winter — would need
  AND/OR logic instead of a pure product; flag as a known simplification

### 2.2 Sub-model: asteroid / comet impact
- Data: NASA/ESA NEO survey size-frequency counts; crater-count record as a cross-check
- Model: marked Poisson process — rate λ(size) fit from survey data, civilization-ending
  lethality probability p(size) as a step/logistic function of impactor diameter
  (Tunguska/Chelyabinsk-scale vs. Chicxulub-scale threshold)
- PyMC sketch:
  ```
  with pm.Model():
      log_rate = pm.Normal("log_rate", mu=..., sigma=...)  # per size bin, from survey data
      threshold = pm.Normal("lethal_diam_km", mu=10, sigma=2)  # weakly informative
      # posterior predictive: P(lethal impact in next t years)
  ```
- This is the best-constrained pathway — treat it as the "sanity check" sub-model

### 2.3 Sub-model: pandemic
- Data: historical pandemic frequency (plague, 1918 flu, COVID) as partial data;
  zero historical events at extinction-level severity
- Model: hierarchical — emergence rate × P(extinction-level | emergence), the latter
  built from R0, case-fatality rate, intervention-effectiveness sub-parameters
- Zero-event tail: use Laplace's rule of succession / Jeffreys prior to bound
  P(extinction-level | emergence) rather than assuming a point value
- Optional informative prior source: Ord (2020, *The Precipice*) elicited estimates,
  used as a prior mean with inflated uncertainty rather than taken at face value

### 2.4 Sub-model: ecological collapse
- Least data-rich pathway — treat as time-*varying* hazard tied to an anthropogenic
  driver trajectory (e.g. logistic-increasing hazard keyed to a warming or
  biodiversity-loss proxy trend) rather than constant-rate
- Priors here are the most subjective in the notebook — say so explicitly and consider
  a structured-elicitation-style prior (a small set of named scenarios with assigned
  weights) rather than a single smooth functional form, so the subjectivity is visible
  and auditable rather than hidden in a parametric choice

### 2.5 Combining the three pathways
- S_catastrophe(t) = S_asteroid(t) · S_pandemic(t) · S_ecological(t)
- Plot combined catastrophe-only survival curve; report combined median/CI
- Diagnostic: report each pathway's marginal contribution to short-horizon risk
  (e.g. next 1,000 / 10,000 / 1M years) since they operate on very different timescales

## 3. Part E — Combining background (v1/v2) with catastrophe (Part D)
- S_total(t) = S_background(t) · S_catastrophe(t), computed separately per reference class
  and per background method (outside-view / Gott / Carter-Leslie)
- Expect catastrophe pathways to dominate the *short*-horizon tail and background hazard
  to dominate the *long*-horizon tail — show this as a single combined-hazard plot with
  both components visible, not just the product curve, so the crossover point is legible

## 4. Part F — Robust Bayes across reference classes
### 4.1 Define the prior class Γ
- Γ = the 3 reference classes × 2 parametric forms (Kaplan-Meier empirical, log-logistic)
  already built in v1/v2 — six posteriors, no new fitting required

### 4.2 Robust interval
- For a quantity of interest Q (e.g. P(T < 1 Myr), or median T), compute
  min over Γ and max over Γ of the posterior value of Q
- Report as an interval, explicitly *not* collapsed to a point — this is the
  "class of priors" / Berger-style sensitivity analysis, distinct from BMA

### 4.3 Soft-BMA comparison (secondary, for contrast)
- Also compute a single blended posterior weighting the 3 reference classes by
  out-of-sample predictive fit (WAIC/LOO) or by phylogenetic proximity to humans
- Present side-by-side with 4.2's interval so the notebook shows both "our best
  single number" and "how much that number depends on an unresolved modeling choice"

## 5. Sensitivity / diagnostics
- Vary each catastrophe sub-model's most subjective prior (esp. ecological collapse)
  and show how much the combined median shifts — same spirit as v2's reference-class
  narrowing sensitivity check (24-40% shift reported there)

## 6. Pros / cons
- Same convention as v1/v2. Cons should explicitly include: independence assumption
  between catastrophe pathways, subjectivity of the ecological-collapse prior, and the
  framing tension between "background" and "foreground" hazard noted in §1

## 7. Conclusions
- Standard summary; headline should report both the combined point estimate(s) and
  the robust interval width from Part F, not just a single number

## 8. References
- Existing v1/v2 bibliography, plus: NASA/ESA NEO survey data sources, Ord (2020)
  *The Precipice*, PRA/fault-tree methodology references (e.g. WASH-1400 lineage),
  Berger — robust Bayesian analysis overview, Walley — imprecise probability

## 9. Appendix — new data provenance
- Document sources/access dates for NEO survey data and any pandemic/ecological
  datasets pulled in, matching the existing PBDB/NOW provenance notes from v1


---------------------

## Me

Please go ahead and build out the asteroid fault-tree sub-model in a Jupyter notebook.

## Claude

Good, `az.extract` is the right API for this arviz version. Let me update the draft.Now let's execute it end-to-end to confirm it runs clean, matching the zero-errors convention from v1/v2.Executed cleanly. Let me verify there are no error outputs embedded and spot-check the key numeric results.Zero errors, 22 cells. Let's move it to outputs and present it.The notebook builds and runs cleanly end-to-end (22 cells, zero errors). It follows your v1/v2 conventions — abstract, pros/cons, conclusions, references — and does the following:

1. **Literature anchor points** — six published size/frequency estimates from bolide-scale (~4m, ~annual) up through Chicxulub-scale (~10km, ~1/100Myr), each with an assumed log-uncertainty.
2. **Bayesian power-law fit** to those anchors: `N(>D) = k·D⁻ᵇ` with priors on `log10(k)` and `b` rather than point estimates.
3. **Lethality threshold** as a logistic function — importantly modeled in *log10(diameter)*, not linear diameter. I hit a real bug building this: a linear-km transition width leaks non-trivial "lethal" probability onto the vastly more numerous small impactors and inflates the hazard by many orders of magnitude. The notebook flags this explicitly rather than quietly fixing it offstage.
4. **Marked Poisson process combination** via Monte Carlo integration over posterior draws, giving a posterior survival function S(t) with a credible band.
5. **Sensitivity check** on the lethality midpoint (5/10/15 km), and a **cross-check** against the ~100 Myr crater-record figure.

Headline result: posterior median recurrence interval for an extinction-level impact of several tens of millions of years, with a wide interval driven mostly by threshold uncertainty rather than the (better-anchored) frequency fit — same order of magnitude as the commonly cited ~100 Myr figure, which is a reasonable outcome for a sketch model.

Next up would be the pandemic and ecological-collapse sub-models — want me to move to one of those next?